In [1]:
import os 
os.environ['SPARK_HOME'] = "/Users/mukesh/opt/spark-3.5.1-bin-hadoop3"
os.environ['JAVA_HOME'] = "/opt/homebrew/opt/openjdk@17/libexec/openjdk.jdk/Contents/Home"
# os.environ['PATH'] is usually handled by these two, but setting it explicitly can't hurt:
os.environ['PATH'] = os.environ['SPARK_HOME'] + "/bin:" + os.environ.get('PATH', '')

In [ ]:
from pyspark.sql import SparkSession
import os
import glob

# Build a list of likely search roots relative to the notebook process cwd
cwd = os.getcwd()
search_roots = [
    cwd,
    os.path.join(cwd, 'jars'),
    os.path.abspath(os.path.join(cwd, '..')),
    os.path.abspath(os.path.join(cwd, '..', 'jars')),
    os.path.abspath(os.path.join(cwd, '..', '..')),
    os.path.abspath(os.path.join(cwd, '..', '..', 'jars')),
    # explicit project root (helpful if kernel cwd is SCD/)
    os.path.abspath(os.path.join(os.path.expanduser('~'), 'Desktop', 'TEST_AGAIN', 'jars'))
]

patterns = ['*mysql*connector*.jar', 'mysql-connector*.jar', '*mysql*.jar']

matches = []
for root in search_roots:
    for pat in patterns:
        matches.extend(glob.glob(os.path.join(root, '**', pat), recursive=True))

# Also fallback: global recursive search under repo root (limited depth)
repo_root = os.path.abspath(os.path.join(cwd, '..'))
matches.extend(glob.glob(os.path.join(repo_root, '**', '*mysql*connector*.jar'), recursive=True))

matches = sorted(set(matches))
mysql_jar = matches[0] if matches else None

if mysql_jar:
    print(f"Found MySQL JDBC driver jar: {mysql_jar}")
    spark = (
        SparkSession.builder.appName("SCD")
        .config("spark.jars", mysql_jar)
        .config("spark.driver.extraClassPath", mysql_jar)
        .getOrCreate()
    )
    print('Configured Spark with jar on driver and executors.')
else:
    print("No MySQL JDBC driver jar found in expected locations. Looked in:")
    for r in search_roots:
        print(' -', r)
    print('You can place the connector jar in one of those folders (e.g. ./jars) and restart the kernel.')
    spark = SparkSession.builder.appName("SCD").getOrCreate()

print("Spark-Version *** :", spark.version)


No MySQL JDBC driver jar found in ./jars or repository root. Place the connector jar in ./jars and restart the kernel, or use spark-submit with --jars.


25/10/31 15:22:08 WARN Utils: Your hostname, mukeshs-MacBook-Pro.local resolves to a loopback address: 127.0.0.1; using 192.168.29.221 instead (on interface en0)
25/10/31 15:22:08 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/10/31 15:22:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
25/10/31 15:22:09 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


Spark-Version *** : 3.5.1


In [3]:
from pyspark.sql.functions import * 
from pyspark.sql.types import * 

source_tbl = """
        CREATE TABLE IF NOT EXISTS  employee_src 
        (
        emp_id INT,
        name VARCHAR(50),
        department VARCHAR(50),
        salary INT,
        effective_date DATE
        );
        """

# emplloyee_trg ( OLAP )
target_tbl = """
        CREATE TABLE IF NOT EXISTS employee_trg
        (
            emp_id INT,
            name VARCHAR(50),
            department VARCHAR(50),
            salary INT,
            effective_start_date DATE,
            effective_end_date DATE,
            flag BOOLEAN
        );

In [4]:
emp_src_schema = StructType([
           StructField("emp_id",IntegerType(),True),
           StructField("name",StringType(),True),
           StructField("department",StringType(),True),
           StructField("salary",IntegerType(),True),
           StructField("effective_date",DateType(),True)
           ])

employee_trg_schema = StructType([
           StructField("emp_id",IntegerType(),True),
           StructField("name",StringType(),True),
           StructField("department",StringType(),True),
           StructField("salary",IntegerType(),True),
           StructField("effective_start_date",DateType(),True),
           StructField("effective_end_date",DateType(),True),
           StructField("flag",BooleanType(),True)
])



In [5]:
from datetime import date

emp_src_data = [
               (1,"Mukesh","HR",5000,date(2025,10,30)),
               (2,"Yash","IT",6000,date(2025,10,30)),
               (3,"Charlie","Finance",8000,date(2025,10,30)),
               ]


src_df = spark.createDataFrame(emp_src_data,emp_src_schema)

src_df.printSchema()

src_df.show()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)



+------+-------+----------+------+--------------+
|emp_id|   name|department|salary|effective_date|
+------+-------+----------+------+--------------+
|     1| Mukesh|        HR|  5000|    2025-10-30|
|     2|   Yash|        IT|  6000|    2025-10-30|
|     3|Charlie|   Finance|  8000|    2025-10-30|
+------+-------+----------+------+--------------+



In [6]:
emp_trg_data = [
                (1,"Mukesh","HR",5000,date(2025,9,1),date(9999,12,31),True),
                (2,"Bob","IT",6000,date(2025,9,15),date(9999,12,31),True),
                (3,"Charlie","Finance",15000,date(2025,9,10),date(9999,12,31),True)
]

trg_df = spark.createDataFrame(emp_trg_data,employee_trg_schema)

trg_df.printSchema()

trg_df.show()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- flag: boolean (nullable = true)

+------+-------+----------+------+--------------------+------------------+----+
|emp_id|   name|department|salary|effective_start_date|effective_end_date|flag|
+------+-------+----------+------+--------------------+------------------+----+
|     1| Mukesh|        HR|  5000|          2025-09-01|        9999-12-31|true|
|     2|    Bob|        IT|  6000|          2025-09-15|        9999-12-31|true|
|     3|Charlie|   Finance| 15000|          2025-09-10|        9999-12-31|true|
+------+-------+----------+------+--------------------+------------------+----+

+------+-------+----------+------+--------------------+------------------+----+
|emp_id|   name|department|salary|effective_start_date|effecti

In [7]:
join_df = src_df.alias("src").join(trg_df.alias("trg"),on="emp_id",how="left")

join_df.printSchema()

join_df.show()

root
 |-- emp_id: integer (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_date: date (nullable = true)
 |-- name: string (nullable = true)
 |-- department: string (nullable = true)
 |-- salary: integer (nullable = true)
 |-- effective_start_date: date (nullable = true)
 |-- effective_end_date: date (nullable = true)
 |-- flag: boolean (nullable = true)

+------+-------+----------+------+--------------+-------+----------+------+--------------------+------------------+----+
|emp_id|   name|department|salary|effective_date|   name|department|salary|effective_start_date|effective_end_date|flag|
+------+-------+----------+------+--------------+-------+----------+------+--------------------+------------------+----+
|     1| Mukesh|        HR|  5000|    2025-10-30| Mukesh|        HR|  5000|          2025-09-01|        9999-12-31|true|
|     2|   Yash|        IT|  6000|    2025-10-30|    Bob| 

In [8]:
changed_df = join_df.filter(
          (col("trg.emp_id").isNull()) | (col("src.name") != col("trg.name"))
).select("src.*")



In [9]:
changed_df.show()

+------+----+----------+------+--------------+
|emp_id|name|department|salary|effective_date|
+------+----+----------+------+--------------+
|     2|Yash|        IT|  6000|    2025-10-30|
+------+----+----------+------+--------------+



In [10]:
from mysql_spark import ConnectDB

In [11]:
HOST = "127.0.0.1"
USER = "root"
PASSWORD = "Paridhi@2019#"  
DATABASE = "dw_poc"
SOURCE_TABLE = "source_data"
TARGET_TABLE = "processed_results"

In [12]:
db = ConnectDB(host=HOST, password=PASSWORD, user=USER, database=DATABASE)

Initialising the database configuration...


In [13]:
 

db.read_mysql_table_to_spark_df(spark, 'employee_src').show()


--- Spark Read: Reading Entire Table 'employee_src' ---
Error reading Spark DataFrame via JDBC: An error occurred while calling o72.jdbc.
: java.lang.ClassNotFoundException: com.mysql.cj.jdbc.Driver
	at java.base/java.net.URLClassLoader.findClass(URLClassLoader.java:445)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:592)
	at java.base/java.lang.ClassLoader.loadClass(ClassLoader.java:525)
	at org.apache.spark.sql.execution.datasources.jdbc.DriverRegistry$.register(DriverRegistry.scala:46)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.$anonfun$driverClass$1$adapted(JDBCOptions.scala:103)
	at scala.Option.foreach(Option.scala:407)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:103)
	at org.apache.spark.sql.execution.datasources.jdbc.JDBCOptions.<init>(JDBCOptions.scala:41)
	at org.apache.spark.sql.execut